# Chat Templates ve Veri Hazırlama — Detaylı Rehber

Bu notebook tüm çıktıları **canlı sunucuda kanıtlanmış** bilgilerle hazırlandı (bkz. `smoke/verify_templates.py`). Hiçbir tahmin yok.

## Neden Önemli?

Chat template ve veri formatı, fine-tuning'in **en sessiz başarısızlık noktası**:
- Yanlış marker → masking sessizce çalışmaz, model bozulur
- Yanlış format → loss hesaplanır ama anlamsız
- Çift `<bos>` → model çıkışı bozuk
- `tools=` parametresi sessizce ignore edilir (Gemma 4!)

Hata mesajı vermez — sadece eğitim kötü çıkar. O yüzden bu konuyu derinlemesine bilmek şart.

## İçindekiler

1. **Chat Template Nedir?** — Jinja → string mantığı
2. **Karşılaştırmalı Render** — 5 template yan yana
3. **`add_generation_prompt`** — generation için ne ekleniyor?
4. **`tools=` Parametresi** — hangi template destekliyor?
5. **`enable_thinking=`** — thinking template'lerinin gerçek davranışı
6. **Multi-turn + System Prompt**
7. **Veri Formatları** — Alpaca / ShareGPT / OpenAI
8. **`standardize_data_formats`** — gerçekte ne yapıyor?
9. **`formatting_prompts_func` Pattern'leri**
10. **Vision Veri Formatı**
11. **Tool Calling Veri Formatı**
12. **Masking Doğrulama** — `train_on_responses_only`
13. **Yaygın Tuzaklar**

## 1. Chat Template Nedir?

Chat template, mesaj listesini (`[{role, content}, ...]`) tek bir **string**'e dönüştüren **Jinja** template'idir.

### Üç Yer
1. **Tokenizer'in kendi içinde** — `tokenizer_config.json` dosyasındaki `chat_template` alanı (HF default)
2. **Unsloth `CHAT_TEMPLATES` dict** — `unsloth/chat_templates.py`'de hard-coded 47 template
3. **`get_chat_template(tokenizer, name)`** — yukarıdaki dict'ten birini seçip tokenizer'a inject eder

### Niye Unsloth'un kendi template'i?
- HF tokenizer'daki template bazen eksik/yanlış olur
- Unsloth'un versiyonu test edilmiş, masking ile uyumlu
- Her model ailesi için optimize edilmiş (örn. Gemma 4'ün `<bos>` quirk'i)

In [ ]:
import unsloth                                # MUTLAKA EN BAŞTA
from unsloth.chat_templates import (
    get_chat_template,
    standardize_data_formats,
    train_on_responses_only,
    CHAT_TEMPLATES,                           # raw dict
)
from transformers import AutoTokenizer
from datasets import Dataset

# Mevcut template'ler
print(f'Toplam {len(CHAT_TEMPLATES)} template kayıtlı')
print(sorted(CHAT_TEMPLATES.keys()))

### Tokenizer'ları Hazırla

5 modeli karşılaştıracağız. Sadece tokenizer yüklüyoruz — model değil. Hızlıdır.

In [ ]:
TOK_FOR = {
    'qwen3-instruct':   'unsloth/Qwen3-4B-Instruct-2507',
    'qwen3-thinking':   'unsloth/Qwen3-4B-Thinking-2507',
    'gemma-4':          'unsloth/gemma-4-E2B-it',
    'gemma-4-thinking': 'unsloth/gemma-4-E2B-it',     # aynı tokenizer, farklı template
    'llama-3.1':        'unsloth/Llama-3.1-8B-Instruct',
}

tokenizers = {}
for name, path in TOK_FOR.items():
    tok = AutoTokenizer.from_pretrained(path)
    tok = get_chat_template(tok, chat_template=name)
    tokenizers[name] = tok
    print(f'OK: {name}')

## 2. Karşılaştırmalı Render

**Aynı konuşma**, 5 farklı template — birebir farklı çıktı. Bu farklar zorunlu, çünkü her model kendi marker'ı ile eğitildi.

In [ ]:
msgs = [
    {'role': 'user',      'content': 'Faiz nedir?'},
    {'role': 'assistant', 'content': 'Faiz, paranin zaman degeridir.'},
]

for name, tok in tokenizers.items():
    print('=' * 60)
    print(name)
    print('=' * 60)
    print(tok.apply_chat_template(msgs, tokenize=False))
    print()

### Marker Karşılaştırması (kanıtlanmış)

| Template | Açılış / Kapanış | Asistan rolü | BOS |
|---|---|---|---|
| `qwen3-instruct` | `<\|im_start\|>` / `<\|im_end\|>` (simetrik) | `assistant` | yok |
| `qwen3-thinking` | `<\|im_start\|>` / `<\|im_end\|>` | `assistant` + auto `<think>` | yok |
| `gemma-4` | **`<\|turn>` / `<turn\|>` (asimetrik!)** | **`model`** | **`<bos>`** |
| `gemma-4-thinking` | `<\|turn>` / `<turn\|>` | `model` + `<\|channel>thought` | `<bos>` |
| `llama-3.1` | `<\|start_header_id\|>` / `<\|eot_id\|>` | `assistant` (system enjekte ediyor!) | `<\|begin_of_text\|>` |

### Gözlem
- **Gemma 4 asimetrik** — Açılış `<\|turn>` ama kapanış `<turn\|>`. Yanlış yazınca masking patlar.
- **Gemma 4 asistan = `model`** (assistant değil!)
- **Llama 3.1 otomatik system prompt** ekliyor — "Cutting Knowledge Date" gibi
- **Qwen3-thinking otomatik `<think>` bloğu** açıyor

## 3. `add_generation_prompt=True`

Inference için zorunlu. Template'in sonuna model'in cevap yazması için **assistant header'ı** ekler. Train'de **gerekmez** (zaten datada var).

**Önemli:** Her template farklı bir şey ekler. İşte gerçek çıktı (kanıtlanmış).

In [ ]:
user_only = [{'role': 'user', 'content': 'Faiz nedir?'}]

for name, tok in tokenizers.items():
    no_gen   = tok.apply_chat_template(user_only, tokenize=False, add_generation_prompt=False)
    with_gen = tok.apply_chat_template(user_only, tokenize=False, add_generation_prompt=True)
    suffix   = with_gen[len(no_gen):]
    print(f'{name:20} appended = {suffix!r}')

### Generation Prompt Eklenenler (kanıtlanmış)

| Template | Eklenen |
|---|---|
| `qwen3-instruct` | `<\|im_start\|>assistant\n` |
| `qwen3-thinking` | **`<\|im_start\|>assistant\n<think>\n`** (auto-thinking!) |
| `gemma-4` | `<\|turn>model\n` |
| `gemma-4-thinking` | **`<\|turn>model\n<\|channel>thought\n<channel\|>`** (custom thinking marker!) |
| `llama-3.1` | `<\|start_header_id\|>assistant<\|end_header_id\|>\n\n` |

### Sonuç
- **Thinking modeller** generation prompt'u kendileri thinking ile başlatıyor
- Bu yüzden model otomatik düşünmeye başlar — sen `<think>` yazmana gerek yok
- Inference'ta `enable_thinking=False` istesen bile çoğu thinking template ignore eder (bkz. Bölüm 5)

## 4. `tools=` Parametresi — KRİTİK

Tool calling (function calling) için template'in `tools` branch'i olmalı. **YOKSA `tools=` argümanı sessizce ignore edilir** — hata bile vermez.

Test edelim:

In [ ]:
tools = [{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': 'Get weather for a city',
        'parameters': {
            'type': 'object',
            'properties': {'city': {'type': 'string'}},
            'required': ['city'],
        },
    },
}]
test_msgs = [{'role': 'user', 'content': 'Istanbul hava nasil?'}]

for name, tok in tokenizers.items():
    without = tok.apply_chat_template(test_msgs, tokenize=False)
    with_t  = tok.apply_chat_template(test_msgs, tokenize=False, tools=tools)
    supported = with_t != without
    diff = len(with_t) - len(without)
    print(f'{name:20} tools= supported = {supported}  (diff: +{diff} chars)')

### Tools Desteği (kanıtlanmış)

| Template | `tools=` Destek | Not |
|---|---|---|
| `qwen3-instruct` | ✅ +600 char | Hermes-style `<tool_call>` |
| `qwen3-thinking` | ✅ +600 char | Aynı, thinking ile |
| **`gemma-4`** | ❌ **0 char** | **Sessizce ignore!** |
| **`gemma-4-thinking`** | ❌ **0 char** | **Sessizce ignore!** |
| `llama-3.1` | ✅ +672 char | JSON function format |

### Pratik Sonuç
- **Tool calling SFT için Qwen3 veya Llama 3.1 kullan**
- Gemma 4 ile tool calling istiyorsan: system prompt'a manuel JSON schema göm + custom `<tool_call>` marker'ı eğit
- Yanılma: `apply_chat_template(messages, tools=[...])` Gemma 4'te hata vermez ama hiçbir şey de yapmaz

## 5. `enable_thinking=` Parametresi

Thinking model'ler için. Davranışı template-bazlı ve **şaşırtıcı**.

In [ ]:
for name in ['qwen3-instruct', 'qwen3-thinking', 'gemma-4', 'gemma-4-thinking']:
    tok = tokenizers[name]
    print(f'\n--- {name} ---')
    for et in [False, True]:
        try:
            out = tok.apply_chat_template(
                [{'role': 'user', 'content': 'Faiz nedir?'}],
                tokenize=False, add_generation_prompt=True,
                enable_thinking=et,
            )
            tail = out[-80:]
            print(f'  enable_thinking={et}: ...{tail!r}')
        except Exception as e:
            print(f'  enable_thinking={et}: FAIL {e}')

### `enable_thinking` Davranışı (kanıtlanmış)

| Template | False | True |
|---|---|---|
| `qwen3-instruct` | ignored | ignored |
| `qwen3-thinking` | **ignored** — her zaman `<think>\n` ekler | aynı |
| `gemma-4` | normal output | **`<\|think\|>\n<turn\|>` ekler önce!** |
| `gemma-4-thinking` | **default thinking marker** kullanır (`<\|channel>thought`) | **alternatif marker** (`<\|think\|>`) |

### Önemli
- `qwen3-thinking` + `enable_thinking=False` → **çalışmaz**, model yine `<think>` ile başlar
- Thinking'i kapatmak için ya **chat template'i değiştir** (qwen3-instruct kullan), ya post-process ile `<think>...</think>` blok'unu sil
- `gemma-4` (non-thinking) bile `enable_thinking=True` ile thinking-style çalıştırılabilir!

## 6. Multi-turn + System Prompt

In [ ]:
multi = [
    {'role': 'system',    'content': 'You are a helpful Turkish assistant.'},
    {'role': 'user',      'content': 'Faiz nedir?'},
    {'role': 'assistant', 'content': 'Faiz, paranin zaman degeridir.'},
    {'role': 'user',      'content': 'Bir ornek ver.'},
]

for name in ['qwen3-instruct', 'gemma-4']:
    print('=' * 60)
    print(name)
    print('=' * 60)
    print(tokenizers[name].apply_chat_template(multi, tokenize=False, add_generation_prompt=True))
    print()

## 7. Veri Formatları

Üç ana format var. Her biri `apply_chat_template`'in beklediği `[{role, content}]` formatına çevrilmeli:

### 7a. Alpaca (Sebastian Raschka ch07, eski format)
```python
{'instruction': 'Define faiz', 'input': '', 'output': 'Faiz parayi zamanin degeridir.'}
```
Manuel çevir → `{role: 'user', content: instruction + (input if any)}` + `{role: 'assistant', content: output}`

### 7b. ShareGPT (eski Alpaca dataset'leri)
```python
{'conversations': [
    {'from': 'human', 'value': 'Define faiz'},
    {'from': 'gpt',   'value': 'Faiz parayi zamanin degeridir.'},
]}
```
→ `standardize_data_formats` otomatik çevirir

### 7c. OpenAI messages (modern, preferred)
```python
{'messages': [
    {'role': 'user',      'content': 'Define faiz'},
    {'role': 'assistant', 'content': 'Faiz parayi zamanin degeridir.'},
]}
```
→ Zaten doğru, dönüştürmeye gerek yok

## 8. `standardize_data_formats` — Gerçek Davranışı

**Source code'a göre:** sadece `conversations` kolonu varsa çalışır. Aksi halde dataset'i değiştirmeden döner.

Aliases:
- system: `['system']`
- user: `['user', 'human', 'input']`
- assistant: `['gpt', 'assistant', 'output']`

Şimdi 3 formatı da test edelim:

In [ ]:
# 8a. Alpaca → unchanged (no 'conversations' col)
alpaca = Dataset.from_list([
    {'instruction': 'Define faiz', 'input': '', 'output': 'Faiz parayi zamanin degeridir.'},
    {'instruction': 'Translate',   'input': 'hello', 'output': 'merhaba'},
])
print('--- Alpaca ---')
print(f'BEFORE cols: {alpaca.column_names}')
out = standardize_data_formats(alpaca)
print(f'AFTER  cols: {out.column_names}')
print(f'Sample[0]:   {out[0]}')
print('=> Alpaca DEĞIŞMEDI (manuel convert lazım)')

In [ ]:
# 8b. ShareGPT → converted (en az 2 satır gerekli, role-detection için)
sharegpt = Dataset.from_list([
    {'conversations': [
        {'from': 'human', 'value': 'Define faiz'},
        {'from': 'gpt',   'value': 'Para zaman degeridir.'},
    ]},
    {'conversations': [
        {'from': 'human', 'value': 'Translate hello'},
        {'from': 'gpt',   'value': 'merhaba'},
    ]},
])
print('--- ShareGPT ---')
print(f'BEFORE: {sharegpt[0]}')
out = standardize_data_formats(sharegpt)
print(f'AFTER:  {out[0]}')
print('=> ShareGPT (from/value/human/gpt) → OpenAI (role/content/user/assistant)')

In [ ]:
# 8c. OpenAI messages → unchanged (no 'conversations' col, çıkmaz)
openai = Dataset.from_list([
    {'messages': [
        {'role': 'user',      'content': 'Define faiz'},
        {'role': 'assistant', 'content': 'Para zaman degeridir.'},
    ]},
])
print('--- OpenAI ---')
print(f'BEFORE cols: {openai.column_names}')
out = standardize_data_formats(openai)
print(f'AFTER  cols: {out.column_names}')
print('=> OpenAI DEĞIŞMEDI (zaten doğru format)')

### Pratik Kural

| Source | `standardize_data_formats` | Pipeline |
|---|---|---|
| Alpaca | hiçbir şey yapmaz | **manuel convert** → `conversations: [{role, content}]` |
| ShareGPT | `{from, value}` → `{role, content}` | direkt çağır, çalışır |
| OpenAI messages | hiçbir şey yapmaz | **kolon adını `conversations` yap** veya `messages` kullan |

## 9. `formatting_prompts_func` Pattern'leri

`SFTConfig(dataset_text_field='text')` derseniz, dataset'te **'text' kolonu** olması gerek. Bu kolonu `formatting_prompts_func` ile oluşturuyoruz.

In [ ]:
# 9a. Qwen3 — standart pattern
tok = tokenizers['qwen3-instruct']

def fmt_qwen3(examples):
    return {'text': [
        tok.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in examples['conversations']
    ]}

# Test
data = Dataset.from_list([
    {'conversations': [
        {'role': 'user', 'content': 'Faiz nedir?'},
        {'role': 'assistant', 'content': 'Para zaman degeridir.'},
    ]},
])
out = data.map(fmt_qwen3, batched=True)
print(out[0]['text'])

In [ ]:
# 9b. Gemma 4 — '<bos>' STRIP zorunlu
# Çift bos → bozuk model. Processor zaten ekleyecek.
tok = tokenizers['gemma-4']

def fmt_gemma4(examples):
    return {'text': [
        tok.apply_chat_template(c, tokenize=False, add_generation_prompt=False).removeprefix('<bos>')
        for c in examples['conversations']
    ]}

out = data.map(fmt_gemma4, batched=True)
print('After Gemma 4 format (no bos at start):')
print(repr(out[0]['text'][:100]))

## 10. Vision Veri Formatı

Vision SFT'de `content` **string değil**, içerik tipleri listesi.

In [ ]:
# Vision conversation — image + text mixed
vision_msg = {
    'messages': [
        {'role': 'user', 'content': [
            {'type': 'text',  'text': 'Bu resimdeki LaTeX nedir?'},
            {'type': 'image', 'image': 'https://example.com/img.png'},  # gerçek PIL.Image olmalı
        ]},
        {'role': 'assistant', 'content': [
            {'type': 'text', 'text': r'\frac{a}{b}'},
        ]},
    ]
}
print('Vision format:')
import json
print(json.dumps(vision_msg, indent=2, ensure_ascii=False))

### Vision Pipeline Notları
1. `Dataset.map(fmt_func)` **kullanma** — Dataset PIL.Image'i pickle edemez. List comprehension kullan:
   ```python
   converted = [convert_to_conversation(s) for s in dataset]
   ```
2. SFTConfig'de **3'lü zorunlu** ayar:
   ```python
   remove_unused_columns = False,
   dataset_text_field = '',
   dataset_kwargs = {'skip_prepare_dataset': True},
   ```
3. Data collator: `UnslothVisionDataCollator(model, tokenizer)`

## 11. Tool Calling Veri Formatı

Bölüm 4'te gördüğümüz gibi sadece `qwen3-*` ve `llama-3.1` `tools=` destekliyor. Eğitim verisi de buna göre.

In [ ]:
# Tool calling SFT için multi-turn assistant→tool→assistant
tool_conv = [
    {'role': 'system', 'content': 'You are a weather assistant.'},
    {'role': 'user', 'content': 'What is the weather in Istanbul?'},
    {'role': 'assistant', 'content': '',
     'tool_calls': [{
         'id': 'call_1', 'type': 'function',
         'function': {'name': 'get_weather', 'arguments': {'city': 'Istanbul'}}
     }]},
    {'role': 'tool', 'tool_call_id': 'call_1', 'name': 'get_weather',
     'content': '{"city": "Istanbul", "temp_c": 18, "condition": "cloudy"}'},
    {'role': 'assistant', 'content': 'It is currently 18°C and cloudy in Istanbul.'},
]

tools = [{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': 'Get weather for a city',
        'parameters': {
            'type': 'object',
            'properties': {'city': {'type': 'string'}},
            'required': ['city'],
        },
    },
}]

tok = tokenizers['qwen3-instruct']
rendered = tok.apply_chat_template(tool_conv, tokenize=False, tools=tools)
print(rendered)

## 12. Masking Doğrulama — `train_on_responses_only`

Bu fonksiyon, training sırasında **sadece asistan cevaplarına loss hesaplamasını** sağlar. Yanlış marker → masking sessizce çalışmaz, full sequence'a loss hesaplar (kötü).

Marker'lar **modele özel:**

| Model | instruction_part | response_part |
|---|---|---|
| Qwen3 (instruct/thinking) | `'<\|im_start\|>user\n'` | `'<\|im_start\|>assistant\n'` |
| Gemma 4 | `'<\|turn>user\n'` | **`'<\|turn>model\n'`** (assistant değil!) |
| Gemma 3 / FunctionGemma | `'<start_of_turn>user\n'` | `'<start_of_turn>model\n'` |
| Llama 3 / 3.1 | `'<\|start_header_id\|>user<\|end_header_id\|>\n\n'` | `'<\|start_header_id\|>assistant<\|end_header_id\|>\n\n'` |

### Doğrulama Pattern'i

```python
# Train sonrası, trainer hazırken:
trainer = train_on_responses_only(
    trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
)

# Doğrula — labels[100] sadece asistan cevabı görmeli
sample = trainer.train_dataset[100]
print('FULL INPUT:')
print(tokenizer.decode(sample['input_ids']))
print('\nUNMASKED LABELS (sadece response gorunmeli):')
print(tokenizer.decode([
    tokenizer.pad_token_id if x == -100 else x
    for x in sample['labels']
]))
```

Eğer **kullanıcı input'u da** unmasked labels'da görünüyorsa → **marker yanlış**. Tekrar kontrol et.

## 13. Yaygın Tuzaklar

| # | Tuzak | Sonuç | Çözüm |
|---|---|---|---|
| 1 | Gemma 4'te `<bos>` strip etmemek | Çift bos, bozuk model | `removeprefix('<bos>')` |
| 2 | Gemma 4'te `assistant` markerı | Masking çalışmaz | `<\|turn>model\n` kullan |
| 3 | `tools=` Gemma 4'e ver | Sessiz ignore | Qwen3/Llama-3.1 kullan, veya manuel system prompt |
| 4 | Vision'da `Dataset.map` | PIL pickle hatası | List comprehension |
| 5 | Vision'da `dataset_text_field='text'` | Data collator format hatası | `dataset_text_field=''` + `skip_prepare_dataset=True` |
| 6 | Qwen3-thinking + `enable_thinking=False` | İgnore — yine `<think>` çıkar | Template değiştir veya post-process |
| 7 | Train'de `add_generation_prompt=True` | Çift assistant header | Train'de **False**, inference'ta True |
| 8 | ShareGPT'te tek satır | role-detection patlar | En az 2 satır + farklı role'ler |
| 9 | Alpaca'yı `standardize_data_formats`'a vermek | Hiçbir şey olmaz | Manuel convert |
| 10 | `trainer.train_dataset[100]['labels']` decode etmemek | Yanlış masking sessizce geçer | Mutlaka decode-and-print |

## Özet

1. **Chat template = Jinja → string** dönüşümü; her modelin kendi marker'ı var
2. **Gemma 4 `tools=` desteklemez** — silently ignore eder
3. **Gemma 4 asistan = `model`** (Qwen3 = `assistant`)
4. **Gemma 4 `<bos>` strip zorunlu** (çift bos = bozuk model)
5. **`standardize_data_formats` sadece ShareGPT'i çevirir** — Alpaca/OpenAI dokunmaz
6. **Vision için 3'lü zorunlu config:** `remove_unused_columns=False, dataset_text_field='', dataset_kwargs={'skip_prepare_dataset': True}`
7. **Tool calling SFT için Qwen3 veya Llama 3.1 kullan**
8. **Masking'i mutlaka decode-and-print ile doğrula** — sessiz başarısızlık çok yaygın

**Test edildi:** `smoke/verify_templates.py` — bu notebook'taki tüm bilgiler canlı sunucuda doğrulandı.